In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(BSgenome))
suppressPackageStartupMessages(library(BSgenome.Hsapiens.UCSC.hg38))

In [ ]:
addArchRGenome("hg38")
addArchRThreads(threads = 8)

projdir   <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
proj_path <- paste0(projdir, 'atac/archr/processed/YS3_intersect_rna')
out_dir   <- paste0(projdir, 'csv/chromvar/')

donors         <- c('C29', 'C37', 'C38', 'C39')
clusters_order <- c('Start', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS',
                    'Sendai', 'Delayed', 'Inflamed', 'Skin')

## 1. Load project and check prerequisites

In [ ]:
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = FALSE)
proj

In [ ]:
cat('Cell metadata columns:\n')
print(names(proj@cellColData))

cat('\nAvailable matrices:\n')
print(getAvailableMatrices(proj))

cat('\nPeak annotation names:\n')
print(names(proj@peakAnnotation))

### load peaks

In [ ]:
peaks_path <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed'
peaks_df <- as.data.frame(fread(peaks_path))
peaks_df <- peaks_df[,1:3]
colnames(peaks_df) <- c('chrom', 'start', 'end')
peaks_granges <- makeGRangesFromDataFrame(peaks_df)
peaks_granges

In [ ]:
proj <- addPeakSet(
  ArchRProj = proj,
  peakSet = peaks_granges,
  genomeAnnotation = getGenomeAnnotation(proj),
  force = FALSE
)

In [ ]:
proj

## 2. Add cell metadata (coarse_v7, donor, coarse_donor)

In [ ]:
clust_path <- paste0(projdir, 'csv/clusters/rna_leiden_v7.csv')
clusters   <- read.csv(clust_path, sep = '\t', row.names = 1)

split_strings      <- strsplit(rownames(clusters), '_')
barcodes           <- sapply(split_strings, `[`, 1)
sample             <- sapply(split_strings, `[`, 2)
clusters$archr_bc  <- paste0(sample, '#', barcodes)
rownames(clusters) <- clusters$archr_bc

clusters$coarse_donor <- paste0(clusters$coarse_v7, '_', clusters$assignment)

proj <- addCellColData(proj, data = clusters$coarse_v7,    name = 'coarse_v7',    cells = rownames(clusters), force = TRUE)
proj <- addCellColData(proj, data = clusters$assignment,   name = 'donor',        cells = rownames(clusters), force = TRUE)
proj <- addCellColData(proj, data = clusters$coarse_donor, name = 'coarse_donor', cells = rownames(clusters), force = TRUE)

meta <- as.data.frame(proj@cellColData)
cat('Cell counts per coarse_v7:\n')
print(table(meta$coarse_v7))

## 3. Add motif annotations (CISBP) if not already present

In [ ]:
if (!'Motif' %in% names(proj@peakAnnotation)) {
    cat('Adding CISBP motif annotations...\n')
    proj <- addMotifAnnotations(
        ArchRProj  = proj,
        motifSet   = 'cisbp',
        name       = 'Motif',
        force      = FALSE
    )
} else {
    cat('Motif annotations already present.\n')
}

cat('\nMotif annotation names:', names(proj@peakAnnotation), '\n')

## 4. Compute chromVAR deviation scores

chromVAR quantifies TF motif accessibility per cell by comparing observed vs. expected insertions in motif-containing peaks. Outputs two matrices:
- **deviations**: z-score of deviation from expectation (per cell x motif)
- **deviation scores** (raw): signed deviation values

Requires a PeakMatrix in the project.

In [ ]:
# Check PeakMatrix exists
available_mats <- getAvailableMatrices(proj)
if (!'PeakMatrix' %in% available_mats) {
    stop('PeakMatrix not found. Run addPeakMatrix() first (see 01i notebook).')
} else {
    cat('PeakMatrix found. Proceeding with chromVAR.\n')
}

In [ ]:
# Add background peaks required by chromVAR
proj <- addBgdPeaks(proj, force = FALSE)

In [ ]:
# Run chromVAR — adds 'MotifMatrix' to the project
proj <- addDeviationsMatrix(
    ArchRProj  = proj,
    peakAnnotation = 'Motif',
    force      = FALSE
)

cat('\nMatrices available after chromVAR:\n')
print(getAvailableMatrices(proj))

## 5. Extract deviation z-scores per cell

In [ ]:
# Pull the MotifMatrix (contains both z-scores and deviations)
# sedev rows = motifs, cols = cells
sedev <- getMatrixFromProject(
    ArchRProj  = proj,
    useMatrix  = 'MotifMatrix',
    verbose    = TRUE
)
sedev

In [ ]:
cat('Assay names:', assayNames(sedev), '\n')
cat('Dimensions (motifs x cells):', dim(sedev), '\n')
cat('First 10 motif names:\n')
print(head(rownames(sedev), 10))

In [ ]:
# Extract z-score matrix (deviations assay is the z-score in ArchR)
zscore_mat <- assay(sedev, 'z')  # motifs x cells
cat('Z-score matrix dimensions:', dim(zscore_mat), '\n')
cat('Range:', range(zscore_mat, na.rm = TRUE), '\n')

In [ ]:
outdir <- paste0(projdir, '/rds/atac/processed/')
saveRDS(zscore_mat, file = file.path(outdir, "chromvar_zscores.rds"))

### write out chromvar deviations

## 6. Pseudobulk deviation scores per coarse_donor group

In [ ]:
# Match cells to their coarse_donor label
cell_meta   <- as.data.frame(proj@cellColData)
cell_meta$cell <- rownames(cell_meta)

# Align to column order of zscore_mat
cell_meta <- cell_meta[colnames(zscore_mat), ]

# Compute mean z-score per group per motif
# groups      <- cell_meta$coarse_donor
groups      <- cell_meta$coarse
group_levels <- unique(groups[!is.na(groups)])

pseudo_list <- lapply(group_levels, function(g) {
    idx  <- which(groups == g)
    if (length(idx) < 10) return(NULL)  # skip tiny groups
    rowMeans(zscore_mat[, idx, drop = FALSE], na.rm = TRUE)
})
names(pseudo_list) <- group_levels
pseudo_list  <- pseudo_list[!sapply(pseudo_list, is.null)]

pseudo_mat   <- do.call(cbind, pseudo_list)  # motifs x groups
cat('Pseudobulk matrix dimensions:', dim(pseudo_mat), '\n')

## 7. TFs of interest — subset and export

In [ ]:
tfs_of_interest <- c(
    'NFE2L2', 'KLF6', 'KLF4', 'ZBTB7A',
    'BNC2', 'ZEB1',
    'CEBPB', 'CEBPD',
    'POU5F1', 'SOX2',
    'BCLAF1', 'NFE2L1'
)

motif_names <- rownames(zscore_mat)
matched <- sapply(tfs_of_interest, function(tf) {
    hits <- grep(paste0('^', tf, '_'), motif_names, value = TRUE, ignore.case = TRUE)
    if (length(hits) == 0) hits <- grep(tf, motif_names, value = TRUE, ignore.case = TRUE)
    hits[1]
})
names(matched) <- tfs_of_interest
cat('Motif matches:\n')
print(matched)

In [ ]:
matched_clean <- matched[!is.na(matched)]

# Long-format pseudobulk deviation df for TFs of interest
pseudo_sub <- pseudo_mat[matched_clean, , drop = FALSE]
rownames(pseudo_sub) <- names(matched_clean)

pseudo_df <- as.data.frame(pseudo_sub) %>%
    rownames_to_column('tf') %>%
    pivot_longer(-tf, names_to = 'group', values_to = 'mean_zscore')

# Parse cluster and donor from group label
pseudo_df$cluster <- sub('_C[0-9]+$', '', pseudo_df$group)
pseudo_df$donor   <- regmatches(pseudo_df$group, regexpr('C[0-9]+$', pseudo_df$group))

pseudo_df$cluster <- factor(pseudo_df$cluster, levels = clusters_order)
pseudo_df$donor   <- factor(pseudo_df$donor,   levels = donors)

head(pseudo_df)

## 8. Visualize: heatmap of chromVAR z-scores (TF x cluster, facet by donor)

In [ ]:
ggplot(pseudo_df, aes(x = cluster, y = tf, fill = mean_zscore)) +
    geom_tile() +
    scale_fill_gradient2(low = 'blue', mid = 'white', high = 'red',
                         midpoint = 0, name = 'Mean\nZ-score') +
    facet_wrap(~donor, nrow = 1) +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          panel.background = element_blank(),
          plot.background  = element_blank()) +
    labs(title = 'chromVAR deviation z-scores (pseudobulk per coarse_donor)',
         x = 'Cluster', y = 'TF')

In [ ]:
# Donor variability: SD across donors within each TF x cluster
variability_df <- pseudo_df %>%
    group_by(tf, cluster) %>%
    summarise(
        mean_z   = mean(mean_zscore, na.rm = TRUE),
        sd_z     = sd(mean_zscore,   na.rm = TRUE),
        n_donors = sum(!is.na(mean_zscore)),
        .groups  = 'drop'
    ) %>%
    arrange(desc(sd_z))

cat('Most donor-variable TF chromVAR scores:\n')
print(head(variability_df, 20))

## 9. Export: all-motif pseudobulk matrix + TF subset long table

In [ ]:
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# Full pseudobulk matrix (all motifs x coarse_donor groups)
pseudo_full_df <- as.data.frame(pseudo_mat) %>%
    rownames_to_column('motif')
fwrite(pseudo_full_df,
       paste0(out_dir, 'chromvar_pseudobulk_coarse_donor.csv'),
       sep = '\t')

# TF-subset long table
fwrite(pseudo_df,
       paste0(out_dir, 'chromvar_tfs_of_interest_coarse_donor.csv'),
       sep = '\t')

# Donor variability table
fwrite(variability_df,
       paste0(out_dir, 'chromvar_donor_variability.csv'),
       sep = '\t')

cat('Outputs written to:', out_dir, '\n')

In [ ]:
sessionInfo()

In [ ]:
Sys.time()